# 02 — Feature engineering, network telemetry join and the quality gate

Builds the analytical frame the model and the star schema use, then runs the
declarative quality checks and inspects the resulting `dq_results` table.

⚠️ The network telemetry is **synthetic** (generated by `data/generate_network_kpis.py`).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from telco_churn import config, ingest, clean, features, quality, model, warehouse

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid", palette=["#117865", "#C0392B", "#1F2A44", "#8A94A6"])

# Real file if present, otherwise the synthetic fixture
RAW = config.DEFAULT_RAW_FILE if config.DEFAULT_RAW_FILE.exists() else config.FIXTURE_RAW_FILE
print("Using:", RAW.name, "| synthetic fixture" if "fixture" in RAW.name else "| real IBM data")

In [ ]:
raw = ingest.load_raw_telco(RAW)
customers, _ = clean.clean_telco(raw)
kpis = ingest.load_network_kpis(config.DEFAULT_KPI_FILE)
print(kpis.shape); kpis.head()

## 1. Customer features

In [ ]:
cust = features.add_customer_features(customers)
cust[["customer_id", "tenure_months", "tenure_bucket", "num_addon_services", "is_auto_payment",
      "lifetime_avg_monthly", "charge_ratio", "revenue_band"]].head(10)

## 2. Network aggregates (12-month window, last-3-month trend, technology-relative quality score)

In [ ]:
feats = features.add_network_features(cust, kpis)
net_cols = ["avg_dropped_call_pct", "avg_latency_ms", "avg_download_mbps", "total_outage_minutes",
            "network_complaints", "dropped_call_trend_pp", "latency_trend_pct", "network_quality_score", "network_quality_band"]
feats[["customer_id", "internet_service", "churn"] + net_cols].head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=feats, x="internet_service", y="avg_latency_ms", hue="churn", ax=axes[0])
axes[0].set_title("Latency by technology — why the score is technology-relative")
order = ["Poor", "Fair", "Good", "Excellent"]
g = feats.groupby("network_quality_band")["churn_flag"].mean().reindex(order) * 100
sns.barplot(x=g.index, y=g.values, ax=axes[1], color="#117865"); axes[1].set_title("Churn rate by quality band (synthetic)")
plt.tight_layout()

## 3. Quality gate

In [ ]:
dq = quality.run_quality_gate(feats, kpis, expected_rows=config.EXPECTED_REAL_ROWS if "fixture" not in RAW.name else None,
                              fail_on_error=False)
display(dq["status"].value_counts())
dq[dq["status"] != "PASS"]

In [ ]:
# Break something on purpose to see the gate react
broken = pd.concat([kpis, kpis.iloc[[0]].assign(customer_id="9999-ZZZZZ")])
dq_broken = quality.run_quality_gate(feats, broken, fail_on_error=False)
dq_broken[dq_broken["status"] == "FAIL"]

## 4. Star schema preview

In [ ]:
tables = warehouse.build_star_schema(feats, kpis)
{k: v.shape for k, v in tables.items()}

In [ ]:
tables["dim_contract"].head(), tables["fact_customer"].head(3)